# 02 · Limpieza (Data Preparation)

**Objetivo:** dejar los indicadores demográficos de WPP2024 listos para cruzarlos con PISA: solo países, solo los **años de los ciclos de PISA**, sin columnas redundantes, con las marcas necesarias para no confundir datos con supuestos, y unidos a `pisa_unificado.csv`.

- Se lee de `data/raw/` (y de `data/processed/pisa_unificado.csv`, que genera `00_unificacion_pisa`) y se escribe en `data/processed/`. **`data/raw/` nunca se modifica.**
- Cada decisión se justifica en una celda de texto y se cuenta cuántas filas afecta.
- Punto de partida: la lista de hallazgos del final de `01_exploracion`.

**Salidas** (en `data/processed/`):

| Archivo | Una fila es... | Para qué |
|---|---|---|
| `wpp_paises_pisa.csv` | un país en un año de PISA (237 países × 9 años) | Contexto demográfico de **todos** los países, participen o no en PISA |
| `wpp_agregados_pisa.csv` | un agregado (Mundo, continente, grupo de ingreso) en un año de PISA | Valores de referencia; **no se suman** con los países |
| `pisa_wpp.csv` | un país en un ciclo de PISA en el que participó | Tabla principal del análisis: PISA + WPP |
| `diccionario_wpp.csv` | una columna de WPP | Descripción, unidad y origen de cada columna |

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))  # permite importar src/

import pandas as pd
import matplotlib.pyplot as plt

from src.utils import cargar_raw, cargar_processed, guardar_processed, resumen
from src.estilo import aplicar_estilo

aplicar_estilo()
pd.set_option("display.max_columns", None)

ULTIMO_ESTIMADO = 2023  # último año estimado de WPP2024; desde 2024 todo es proyección

### Criterios (de `01_exploracion` y de los principios éticos de la ASA)

1. **Sin doble conteo:** países y agregados van en archivos distintos.
2. **Años fijados de antemano:** solo los ciclos de PISA. No se eligen otros años según convenga al resultado.
3. **Hechos vs. supuestos:** toda fila lleva `es_proyeccion`. PISA 2025 cae en años proyectados y se acompaña de sus intervalos de predicción.
4. **Unidades sin cambiar:** se conservan las de la ONU (conteos en **miles**) y el nombre original de cada columna, para poder rastrearla hasta el diccionario oficial. La unidad de cada columna queda en `diccionario_wpp.csv`.
5. **Transparencia:** cada filtro imprime cuántas filas quita y por qué.
6. **Reproducibilidad:** todo sale de los archivos originales; no hay pasos manuales ni aleatorios.

## 1. Carga

In [ ]:
dem_raw = cargar_raw(
    "WPP2024_Demographic_Indicators_Medium_csv.gz",
    compression="gzip",
    encoding="utf-8-sig",
    low_memory=False,
)
sup_raw = cargar_raw("WPP2024_Supplement_Births_and_LE60.csv", encoding="utf-8-sig")
var_raw = cargar_raw(
    "WPP2024_Demographic_Indicators_OtherVariants_csv.gz",
    compression="gzip",
    encoding="utf-8-sig",
    low_memory=False,
)
dic = cargar_raw("WPP2024_Demographic_Indicators_notes.csv", encoding="utf-8-sig")
pisa = cargar_processed("pisa_unificado.csv")

# Se trabaja siempre sobre copias
dem, sup, var = dem_raw.copy(), sup_raw.copy(), var_raw.copy()
for nombre, d in [
    ("Medium", dem),
    ("Supplement", sup),
    ("OtherVariants", var),
    ("PISA", pisa),
]:
    print(f"{nombre:14s} {d.shape}")

## 2. Años: solo los ciclos de PISA

Los años se toman de `pisa_unificado.csv`, no se escriben a mano, y se comprueba que son los ciclos oficiales. Así WPP y PISA quedan siempre alineados aunque se añada un ciclo nuevo.

- 2000-2022 son **estimaciones** de la ONU.
- 2025 es **proyección** (variante media); se le añaden los intervalos de predicción en la sección 7.

In [ ]:
PISA_ANIOS = sorted(pisa["Time"].unique().tolist())
assert PISA_ANIOS == [2000, 2003, 2006, 2009, 2012, 2015, 2018, 2022, 2025], PISA_ANIOS
print("Ciclos de PISA:", PISA_ANIOS)
print("Proyectados en WPP:", [a for a in PISA_ANIOS if a > ULTIMO_ESTIMADO])

In [ ]:
antes = len(dem)
dem = dem.loc[dem["Time"].isin(PISA_ANIOS)]
sup = sup.loc[sup["Time"].isin(PISA_ANIOS)]
var = var.loc[var["Time"].isin(PISA_ANIOS)]
print(
    f"Medium: {antes:,} -> {len(dem):,} filas ({len(PISA_ANIOS)} años de {dem_raw['Time'].nunique()})"
)
print("El año 2101 (casi vacío) queda fuera con este filtro.")

## 3. Países y agregados por separado

`Location` mezcla 237 países con agregados. Si se mezclan, cualquier suma o promedio cuenta la misma población varias veces (en 2023, unas 45 veces).

- **Países:** `LocTypeName == "Country/Area"`. Se comprueba que es lo mismo que `ISO3_code.notna()`.
- **Agregados de referencia:** se guardan aparte solo el Mundo, las regiones geográficas y los grupos de ingreso. Los demás tipos (grupos ad hoc, regiones ODS, subregiones...) no se usan en este análisis. Las regiones geográficas se solapan entre sí (`Americas` = `Latin America and the Caribbean` + `Northern America`), así que tampoco se suman.

In [ ]:
es_pais = dem["LocTypeName"].eq("Country/Area")
assert es_pais.equals(dem["ISO3_code"].notna()), "LocTypeName e ISO3_code no coinciden"

paises = dem.loc[es_pais].copy()
TIPOS_AGREGADO = ["World", "Geographic region", "Income group"]
agregados = dem.loc[dem["LocTypeName"].isin(TIPOS_AGREGADO)].copy()

print(
    f"Países:    {paises['ISO3_code'].nunique()} × {paises['Time'].nunique()} años = {len(paises):,} filas"
)
print(
    f"Agregados: {agregados['Location'].nunique()} ubicaciones = {len(agregados):,} filas"
)
print(
    f"Descartados (otros agregados): {len(dem) - len(paises) - len(agregados):,} filas"
)
agregados.groupby("LocTypeName")["Location"].unique()

### Agregados que se solapan

Dentro de `wpp_agregados_pisa.csv` tampoco se pueden sumar filas cualesquiera:

- `Americas` = `Latin America and the Caribbean` + `Northern America`.
- Grupos de ingreso: `High-income countries` está dentro de `High-and-upper-middle-income countries`, y `Middle-income countries` = `Upper-middle` + `Lower-middle`.

Se añade la columna `particion`, que marca dos conjuntos **sin solapamiento** que suman el Mundo (se comprueba). Las filas con `particion` vacía solo sirven como referencia individual, nunca para sumar o apilar.

Excepción comprobada: **Guadalupe** no está en ningún grupo de ingreso de WPP (ni en `No income group available`). La partición `ingreso` suma el Mundo menos Guadalupe, unas 400 000 personas (menos del 0,01 %).

| `particion` | Agregados |
|---|---|
| `continentes` | Africa, Asia, Europe, Latin America and the Caribbean, Northern America, Oceania |
| `ingreso` | High-income, Upper-middle-income, Lower-middle-income, Low-income, No income group available |

In [ ]:
PARTICIONES = {
    "continentes": [
        "Africa",
        "Asia",
        "Europe",
        "Latin America and the Caribbean",
        "Northern America",
        "Oceania",
    ],
    "ingreso": [
        "High-income countries",
        "Upper-middle-income countries",
        "Lower-middle-income countries",
        "Low-income countries",
        "No income group available",
    ],
}
agregados["particion"] = pd.NA
for nombre, miembros in PARTICIONES.items():
    assert set(miembros) <= set(agregados["Location"]), set(miembros) - set(
        agregados["Location"]
    )
    agregados.loc[agregados["Location"].isin(miembros), "particion"] = nombre

# Cada partición debe sumar la población mundial en todos los años
# (la de ingreso, sin Guadalupe, que WPP no clasifica por ingreso)
mundo_pob = agregados.loc[agregados["LocTypeName"] == "World"].set_index("Time")[
    "TPopulation1July"
]
guadalupe = paises.loc[paises["ISO3_code"] == "GLP"].set_index("Time")[
    "TPopulation1July"
]
esperado = {"continentes": mundo_pob, "ingreso": mundo_pob - guadalupe}
for nombre in PARTICIONES:
    suma = (
        agregados.loc[agregados["particion"] == nombre]
        .groupby("Time")["TPopulation1July"]
        .sum()
    )
    dif = (suma - esperado[nombre]).abs().max()
    assert dif < 1, f"{nombre} no suma lo esperado (dif. {dif:.1f} miles)"
    print(f"{nombre:12s} cuadra (dif. máx. {dif:.3f} miles)")

agregados[["LocTypeName", "Location", "particion"]].drop_duplicates().sort_values(
    ["particion", "Location"], na_position="last"
)

## 4. Selección de columnas

El Medium trae 54 indicadores, muchos casi idénticos entre sí (correlación > 0,95 en `01_exploracion`). Para el contexto de PISA me quedo con un conjunto pequeño, uno o dos por tema:

| Tema | Columnas | Por qué |
|---|---|---|
| Población | `TPopulation1July`, `PopDensity`, `MedianAgePop`, `PopGrowthRate` | Tamaño, densidad, envejecimiento y crecimiento |
| Fecundidad | `TFR`, `CBR`, `Births`, `Births1519`, `MAC` | Nivel de fecundidad, natalidad, maternidad adolescente y edad media a la maternidad |
| Mortalidad | `LEx`, `LExMale`, `LExFemale`, `IMR`, `Q5`, `CDR` | Esperanza de vida (total y por sexo) y mortalidad infantil |
| Migración | `NetMigrations`, `CNMR` | Saldo migratorio absoluto y relativo |

**Descartadas:**
- Identificadores sin uso en países (`SortOrder`, `SDMX_code`, `LocTypeID`, `ParentID`, `VarID`, `Variant`, `Notes`) e `ISO2_code` (sobra con `ISO3_code`; además, Namibia tiene "NA" y pandas lo lee como nulo).
- `DoublingTime`: está vacío cuando el crecimiento es de 0,5 % anual o menos.
- `TPopulation1Jan` y las versiones por sexo de la población, las muertes y las probabilidades `Q`: son redundantes.
- `LE15`, `LE65` y `LE80`: el Supplement aporta `LE60`, que mide lo mismo.
- `SRB` y `PopSexRatio`: no son relevantes para PISA y se distorsionan en territorios muy pequeños.

In [ ]:
CLAVES = ["LocID", "ISO3_code", "Location", "Time"]
INDICADORES = [
    "TPopulation1July",
    "PopDensity",
    "MedianAgePop",
    "PopGrowthRate",
    "TFR",
    "CBR",
    "Births",
    "Births1519",
    "MAC",
    "LEx",
    "LExMale",
    "LExFemale",
    "IMR",
    "Q5",
    "CDR",
    "NetMigrations",
    "CNMR",
]
assert set(INDICADORES) <= set(dic["Indicator"])

descartadas = [c for c in dem.columns if c not in CLAVES + INDICADORES]
print(
    f"Se conservan {len(INDICADORES)} indicadores; se descartan {len(descartadas)} columnas:"
)
print(descartadas)

paises = paises[CLAVES + INDICADORES]
agregados = agregados[
    ["LocID", "Location", "LocTypeName", "particion", "Time"] + INDICADORES
]

## 5. Unión con el Supplement

Del Supplement solo se añade la esperanza de vida a los 60 años (`LE60` y por sexo). Los nacimientos por sexo no aportan: su suma es `Births` (se comprueba antes de descartarlos).

La unión es uno a uno por `ISO3_code` y `Time`. El Vaticano (`VAT`) no está en el Supplement: se queda con `LE60` nulo, no se imputa. No participa en PISA.

In [ ]:
dif = (sup["BirthsMale"] + sup["BirthsFemale"]).values - paises.set_index(
    ["ISO3_code", "Time"]
).loc[sup.set_index(["ISO3_code", "Time"]).index, "Births"].values
assert abs(dif).max() < 0.01, "BirthsMale + BirthsFemale no reproduce Births"

COLS_SUP = ["LE60", "LE60Male", "LE60Female"]
antes = len(paises)
paises = paises.merge(
    sup[["ISO3_code", "Time"] + COLS_SUP],
    on=["ISO3_code", "Time"],
    how="left",
    validate="one_to_one",
)
assert len(paises) == antes
print("Países sin Supplement:", paises.loc[paises["LE60"].isna(), "ISO3_code"].unique())

## 6. Variables derivadas y marcas

| Columna | Definición | Por qué |
|---|---|---|
| `es_proyeccion` | `Time > 2023` | Distinguir estimaciones de supuestos (en los años de PISA, solo 2025) |
| `territorio_pequeno` | `TPopulation1July < 100` (menos de 100 000 habitantes) | En territorios muy pequeños, el redondeo a miles distorsiona cocientes. Se marcan, no se eliminan |
| `pct_births_adolescentes` | `100 × Births1519 / Births` | % de nacimientos de madres de 15 a 19 años. `Births1519` es un conteo y no se puede comparar entre países de distinto tamaño |
| `participa_pisa` | El país tiene fila en `pisa_unificado.csv` ese año | Comparar participantes y no participantes (sesgo de selección, sección 10) |

In [ ]:
pisa_claves = pisa[["ISO3_code", "Time"]].drop_duplicates()

for d in (paises, agregados):
    d.insert(
        d.columns.get_loc("Time") + 1, "es_proyeccion", d["Time"] > ULTIMO_ESTIMADO
    )
    d["pct_births_adolescentes"] = 100 * d["Births1519"] / d["Births"]

paises["territorio_pequeno"] = paises["TPopulation1July"] < 100
paises = paises.merge(
    pisa_claves.assign(participa_pisa=True), on=["ISO3_code", "Time"], how="left"
)
paises["participa_pisa"] = paises["participa_pisa"].fillna(False).astype(bool)

print(
    "Filas proyectadas:",
    paises["es_proyeccion"].sum(),
    "(año",
    paises.loc[paises["es_proyeccion"], "Time"].unique(),
    ")",
)
print(
    "Territorios pequeños que participan en PISA:",
    paises.loc[
        paises["territorio_pequeno"] & paises["participa_pisa"], "Location"
    ].unique(),
)
print("Filas país-año que participan en PISA:", paises["participa_pisa"].sum())

## 7. Incertidumbre de 2025

PISA 2025 se cruza con datos **proyectados**. Para no presentarlos como una certeza, se añaden los intervalos de predicción del 80 % y del 95 % de `OtherVariants` para la población, la fecundidad y la esperanza de vida.

- Solo existen para 2025; en los años estimados quedan vacíos, porque no hay intervalo de predicción.
- `Median PI` no se añade: coincide con el Medium.

In [ ]:
PI = {
    "Lower 80 PI": "PI80_inf",
    "Upper 80 PI": "PI80_sup",
    "Lower 95 PI": "PI95_inf",
    "Upper 95 PI": "PI95_sup",
}
COLS_PI = ["TPopulation1July", "TFR", "LEx"]

pi = var.loc[
    var["Variant"].isin(PI) & var["ISO3_code"].notna(),
    ["ISO3_code", "Time", "Variant"] + COLS_PI,
].pivot_table(index=["ISO3_code", "Time"], columns="Variant", values=COLS_PI)
pi.columns = [f"{ind}_{PI[v]}" for ind, v in pi.columns]
pi = pi[
    [
        f"{ind}_{suf}"
        for ind in COLS_PI
        for suf in ["PI95_inf", "PI80_inf", "PI80_sup", "PI95_sup"]
    ]
].reset_index()
print(
    "Años con intervalos:", pi["Time"].unique(), "| países:", pi["ISO3_code"].nunique()
)

# Coherencia: inf95 <= inf80 <= Medium <= sup80 <= sup95
chk = paises.merge(pi, on=["ISO3_code", "Time"])
for ind in COLS_PI:
    orden = chk[
        [
            f"{ind}_PI95_inf",
            f"{ind}_PI80_inf",
            ind,
            f"{ind}_PI80_sup",
            f"{ind}_PI95_sup",
        ]
    ]
    assert (
        (orden.diff(axis=1).iloc[:, 1:] >= -1e-6).all().all()
    ), f"Intervalos desordenados en {ind}"

paises = paises.merge(pi, on=["ISO3_code", "Time"], how="left", validate="one_to_one")
paises.loc[
    paises["Time"] == 2025,
    ["Location"] + [c for c in paises.columns if c.startswith("TFR")],
].head()

## 8. Tipos, duplicados y nulos

Se conservan los nombres originales de la ONU (en vez de pasarlos a minúsculas) para poder rastrear cada columna hasta el diccionario oficial.

| Columna | Nulos | Decisión | Motivo |
|---|---|---|---|
| `LE60`, `LE60Male`, `LE60Female` | 9 (Vaticano) | Dejar | No está en el Supplement y no participa en PISA |
| `*_PI80_*`, `*_PI95_*` | Todos los años salvo 2025 | Dejar | No hay intervalo de predicción para años estimados |

In [ ]:
paises["Time"] = paises["Time"].astype(int)
paises["ISO3_code"] = paises["ISO3_code"].astype("string")
paises["Location"] = paises["Location"].astype("string")

assert not paises.duplicated(["ISO3_code", "Time"]).any()
assert not agregados.duplicated(["LocID", "Time"]).any()

nulos = resumen(paises).query("nulos > 0")
cols_pi = [c for c in paises.columns if "_PI" in c]
assert set(nulos.index) <= set(COLS_SUP + cols_pi), set(nulos.index) - set(
    COLS_SUP + cols_pi
)
assert paises.loc[paises["Time"] == 2025, cols_pi].notna().all().all()
nulos

## 9. Outliers

En `01_exploracion` los valores extremos resultaron reales: microestados (densidad), países del Golfo (migración), conflictos (mortalidad). **No se elimina ninguno.** Se revisan los de los países que participan en PISA para tenerlos presentes al interpretar.

In [ ]:
en_pisa = paises.loc[paises["participa_pisa"]]
en_pisa[INDICADORES + COLS_SUP + ["pct_births_adolescentes"]].plot(
    kind="box", subplots=True, layout=(-1, 5), figsize=(16, 12), sharex=False
)
plt.tight_layout()

In [ ]:
for col in ["CNMR", "PopDensity", "PopGrowthRate", "IMR", "TFR"]:
    print(f"--- {col}")
    display(en_pisa.nlargest(3, col)[["Location", "Time", col]])

## 10. Unión con PISA

- Unión uno a uno por `ISO3_code` y `Time`, solo con los países-año que participaron en PISA.
- **`OED` (promedio OCDE) se excluye:** no existe en WPP. Para calcularlo habría que promediar los países miembros de cada año, y la lista cambia entre ciclos. El promedio OCDE de PISA sigue disponible en `pisa_unificado.csv`.

In [ ]:
print("Filas de PISA:", len(pisa))
pisa_wpp = pisa.merge(
    paises.drop(columns="participa_pisa"),
    on=["ISO3_code", "Time"],
    how="inner",
    validate="one_to_one",
)
excluidas = pisa.loc[
    ~pisa.set_index(["ISO3_code", "Time"]).index.isin(
        pisa_wpp.set_index(["ISO3_code", "Time"]).index
    )
]
print(
    f"Unidas: {len(pisa_wpp)} | excluidas: {len(excluidas)} -> {excluidas['ISO3_code'].unique()}"
)
assert set(excluidas["ISO3_code"]) == {"OED"}

# Orden: claves y marcas primero, luego PISA, luego WPP
primeras = [
    "ISO3_code",
    "LocID",
    "Location",
    "Time",
    "es_proyeccion",
    "territorio_pequeno",
]
pisa_wpp = pisa_wpp[primeras + [c for c in pisa_wpp.columns if c not in primeras]]
pisa_wpp.groupby("Time").agg(
    paises=("ISO3_code", "size"), proyectado=("es_proyeccion", "first")
)

### Participantes con cobertura parcial

En algunos ciclos, PISA no evaluó a todo el país sino a una ciudad o a algunas regiones. `00_unificacion_pisa` les asignó el código del país, como hace el Banco Mundial, así que la unión les pega la demografía **de todo el país**. No es doble conteo, pero mezcla unidades distintas: por ejemplo, los puntajes de cuatro provincias ricas de China con la fecundidad y la mortalidad infantil de toda China.

Nombres comprobados en las tablas de la OCDE y en `00_unificacion_pisa`:

| País | Ciclos | Entidad evaluada |
|---|---|---|
| CHN | 2009, 2012 | Shanghái |
| CHN | 2015 | Pekín, Shanghái, Jiangsu y Cantón (B-S-J-G) |
| CHN | 2018, 2025 | Pekín, Shanghái, Jiangsu y Zhejiang (B-S-J-Z) |
| AZE | 2022 | Bakú (en 2006, 2009 y 2025, nacional) |
| UKR | 2022, 2025 | 18 y 17 de 27 regiones (en 2018, nacional) |
| TJK | 2025 | Dusambé |
| IRQ | 2025 | Región del Kurdistán |

**Decisión:** se **marcan**, no se eliminan, con `cobertura_pisa_parcial` y `entidad_pisa`. Al relacionar PISA con la demografía hay que **excluirlas** (`~cobertura_pisa_parcial`). Para describir solo los resultados de PISA se pueden mantener, indicando qué entidad se evaluó.

In [ ]:
COBERTURA_PARCIAL = {
    ("CHN", 2009): "Shanghái",
    ("CHN", 2012): "Shanghái",
    ("CHN", 2015): "Pekín, Shanghái, Jiangsu y Cantón (B-S-J-G)",
    ("CHN", 2018): "Pekín, Shanghái, Jiangsu y Zhejiang (B-S-J-Z)",
    ("CHN", 2025): "Pekín, Shanghái, Jiangsu y Zhejiang (B-S-J-Z)",
    ("AZE", 2022): "Bakú",
    ("UKR", 2022): "18 de 27 regiones",
    ("UKR", 2025): "17 de 27 regiones",
    ("TJK", 2025): "Dusambé",
    ("IRQ", 2025): "Región del Kurdistán",
}
claves_pw = list(zip(pisa_wpp["ISO3_code"], pisa_wpp["Time"]))
assert set(COBERTURA_PARCIAL) <= set(claves_pw), set(COBERTURA_PARCIAL) - set(claves_pw)

pisa_wpp["cobertura_pisa_parcial"] = [k in COBERTURA_PARCIAL for k in claves_pw]
pisa_wpp["entidad_pisa"] = [COBERTURA_PARCIAL.get(k, "Nacional") for k in claves_pw]

primeras = [
    "ISO3_code",
    "LocID",
    "Location",
    "Time",
    "es_proyeccion",
    "territorio_pequeno",
    "cobertura_pisa_parcial",
    "entidad_pisa",
]
pisa_wpp = pisa_wpp[primeras + [c for c in pisa_wpp.columns if c not in primeras]]

print(
    "Filas con cobertura parcial:",
    pisa_wpp["cobertura_pisa_parcial"].sum(),
    "de",
    len(pisa_wpp),
)
pisa_wpp.loc[
    pisa_wpp["cobertura_pisa_parcial"],
    ["Location", "Time", "entidad_pisa", "PISA_Math_Total"],
]

### Sesgo de selección

Los países de PISA no son una muestra al azar del mundo: participan sobre todo los de ingreso medio y alto. Las conclusiones de `pisa_wpp.csv` valen para los participantes, no para todos los países. Como referencia, en 2022:

In [ ]:
(
    paises.loc[paises["Time"] == 2022]
    .groupby("participa_pisa")[
        ["TFR", "LEx", "IMR", "MedianAgePop", "pct_births_adolescentes"]
    ]
    .median()
    .round(1)
    .rename(index={True: "Participa en PISA", False: "No participa"})
)

## 11. Diccionario de columnas

Una fila por columna de WPP en los archivos de salida, con su descripción, unidad y origen. Las de PISA están documentadas en `00_unificacion_pisa`.

In [ ]:
desc = dic.set_index("Indicator")[["IndicatorName", "Unit"]].rename(
    columns={"IndicatorName": "descripcion", "Unit": "unidad"}
)
filas = [
    ("ISO3_code", "Código ISO 3166-1 alfa-3 del país", "", "WPP2024 Medium"),
    ("LocID", "Código numérico de ubicación de la ONU (M49)", "", "WPP2024 Medium"),
    ("Location", "Nombre del país o agregado (inglés)", "", "WPP2024 Medium"),
    (
        "LocTypeName",
        "Tipo de agregado (solo en wpp_agregados_pisa)",
        "",
        "WPP2024 Medium",
    ),
    ("Time", "Año (solo ciclos de PISA)", "año", "WPP2024 Medium"),
    (
        "es_proyeccion",
        f"True si Time > {ULTIMO_ESTIMADO}: dato proyectado, no estimado",
        "booleano",
        "Derivada",
    ),
]
filas += [
    (c, desc.loc[c, "descripcion"], desc.loc[c, "unidad"], "WPP2024 Medium")
    for c in INDICADORES
]
filas += [
    (
        "LE60",
        "Esperanza de vida a los 60 años, ambos sexos",
        "years",
        "WPP2024 Supplement",
    ),
    (
        "LE60Male",
        "Esperanza de vida a los 60 años, hombres",
        "years",
        "WPP2024 Supplement",
    ),
    (
        "LE60Female",
        "Esperanza de vida a los 60 años, mujeres",
        "years",
        "WPP2024 Supplement",
    ),
    (
        "pct_births_adolescentes",
        "100 × Births1519 / Births: % de nacimientos de madres de 15 a 19 años",
        "percentage",
        "Derivada",
    ),
    (
        "territorio_pequeno",
        "True si TPopulation1July < 100 (menos de 100 000 habitantes)",
        "booleano",
        "Derivada",
    ),
    (
        "participa_pisa",
        "True si el país tiene datos de PISA ese año (solo en wpp_paises_pisa)",
        "booleano",
        "Derivada",
    ),
    (
        "particion",
        "Conjunto de agregados sin solapamiento que suma el Mundo: 'continentes', 'ingreso' o vacío (no sumar). Solo en wpp_agregados_pisa",
        "",
        "Derivada",
    ),
    (
        "cobertura_pisa_parcial",
        "True si PISA evaluó solo una ciudad o algunas regiones del país ese ciclo. Solo en pisa_wpp",
        "booleano",
        "Derivada",
    ),
    (
        "entidad_pisa",
        "Entidad que evaluó PISA ('Nacional' o el nombre de la ciudad o regiones). Solo en pisa_wpp",
        "",
        "Derivada",
    ),
]
for ind in COLS_PI:
    for suf, texto in [
        ("PI80_inf", "límite inferior del intervalo de predicción del 80 %"),
        ("PI80_sup", "límite superior del intervalo de predicción del 80 %"),
        ("PI95_inf", "límite inferior del intervalo de predicción del 95 %"),
        ("PI95_sup", "límite superior del intervalo de predicción del 95 %"),
    ]:
        filas.append(
            (
                f"{ind}_{suf}",
                f"{ind}: {texto} (solo años proyectados)",
                desc.loc[ind, "unidad"],
                "WPP2024 OtherVariants",
            )
        )

diccionario = pd.DataFrame(
    filas, columns=["columna", "descripcion", "unidad", "origen"]
)
cols_wpp_pw = [c for c in pisa_wpp.columns if c not in pisa.columns]
faltan = (set(paises.columns) | set(agregados.columns) | set(cols_wpp_pw)) - set(
    diccionario["columna"]
)
assert not faltan, faltan
diccionario

## 12. Comprobaciones finales y guardado

In [ ]:
assert (
    paises["Time"].isin(PISA_ANIOS).all() and agregados["Time"].isin(PISA_ANIOS).all()
)
assert pisa_wpp["Time"].isin(PISA_ANIOS).all()
assert paises["ISO3_code"].notna().all() and paises["ISO3_code"].nunique() == 237
assert set(agregados["LocTypeName"]) == set(TIPOS_AGREGADO)
assert pisa_wpp[INDICADORES].notna().all().all()
assert not pisa_wpp.duplicated(["ISO3_code", "Time"]).any()
assert pisa_wpp["cobertura_pisa_parcial"].sum() == len(COBERTURA_PARCIAL)

print(
    f"Medium:   {len(dem_raw):,} filas -> países {len(paises):,} + agregados {len(agregados):,}"
)
print(f"PISA:     {len(pisa):,} filas -> pisa_wpp {len(pisa_wpp):,} (sin OED)")

In [ ]:
guardar_processed(paises, "wpp_paises_pisa.csv")
guardar_processed(agregados, "wpp_agregados_pisa.csv")
guardar_processed(pisa_wpp, "pisa_wpp.csv")
guardar_processed(diccionario, "diccionario_wpp.csv")